# Inspectable service requests

Original implementation material: Apache-2.0. Original prose and synthetic inputs: CC BY-SA 4.0. Read the Python and HTTP Primers first. This notebook requires a local POSIX CPU kernel with the pinned project dependencies. It creates only a loopback server and temporary fictional state; no accounts, model weights or network model service. Every code cell is executed in order by the author runner. The static Chapter 54 contains the complete mechanism and outcomes. Assigned delays are fault controls, not measured model speed.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
assert (root / "code/knowledge-assistant").exists(), "Start the kernel at the book repository root"
sys.path.insert(0, str(root / "code/knowledge-assistant"))
sys.path.insert(0, str(root / "code/part-ix"))
from service import server
from support import QUERY, submit, http


## A checked answer, a historical answer and a failed lookup
Change only the applicable date before comparing amounts. Reading a result does not execute an order action.

In [2]:
with server(rate=100, burst=50) as (app, base):
    current = submit(app, base)
    historical = submit(app, base, {**QUERY, "on_date": "2025-12-31"})
    missing = submit(app, base, {**QUERY, "question": "volcano"})
    assert current.result["answer"]["amount_yuan"] == 750
    assert historical.result["answer"]["amount_yuan"] == 600
    assert missing.state == "abstained"
    print([(j.state, j.result.get("answer", {}).get("amount_yuan")) for j in [current, historical, missing]])
    print(http(base, f"/api/requests/{current.id}", key="demo-south")[0])


[('answered', 750), ('answered', 600), ('abstained', None)]
404


## Cancellation and a stable terminal state
The host must acknowledge stopping, not merely hide the answer. This controlled delay makes the branch observable without needing a slow model.

In [3]:
with server() as (app, base):
    app.fault = {"delay": 0.3}
    job = submit(app, base, wait=False)
    print(http(base, f"/api/requests/{job.id}/cancel", {})[0])
    assert job.done.wait(3)
    assert job.state == "cancelled"
    assert not app.cache.rows
    print([event["stage"] for event in job.events])


202
['accepted', 'cancel_requested', 'running', 'cancelled']


## Transfer exercise and visible answer
If the final result arrives before cancellation, should the host rewrite it as cancelled? No: the observable response is 409 with the existing terminal state. Clearing a view can still be undone locally; it does not reverse a server request or order. The behavior suite exercises both race outcomes.